# Single-Prompt Generation Comparison

This notebook runs the same SD1.5 implementations as the long COCO experiment on one custom prompt. All methods reuse the exact same initial latent tensor.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# The metric checkpoint stays outside this code repository.
SSCD_MODEL_PATH = Path(os.environ.get(
    'SSCD_MODEL_PATH',
    ROOT.parent / 'Pullback evaluation' / 'models' / 'sscd_disc_mixup.torchscript.pt',
))
os.environ.setdefault('SSCD_MODEL_PATH', str(SSCD_MODEL_PATH))

MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
PROMPT = 'A red fox standing in an autumn forest, photorealistic, highly detailed, 8k'
NEGATIVE_PROMPT = ''
SELECTED_METHODS = ['clean_ddim', 'cads', 'tpso', 'rho_star']

NUM_PARTICLES = 5
HEIGHT = WIDTH = 512
DDIM_STEPS = 50
GUIDANCE_SCALE = 7.5
ETA = 0.0
INITIAL_SEED = 12345
ETA_SEED = 20800

PULLBACK_RANK = 40
PULLBACK_ITERATIONS = 1
BASIS_TIMESTEP = 500
PROBE_TIMESTEP = 699
CANDIDATE_RHOS = [0.10, 0.125, 0.15, 0.175, 0.20, 0.25]
MAX_CLIP_DROP = 0.5

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from evaluation.metrics import PromptMetrics
from generation import ddim, model
from methods import cads, rho_star, tpso
from methods.adaptive_pullback import sample_adaptive_pullback
from pullback import basis as pullback_basis
from pullback import directions

model.load_model(MODEL_ID, local_files_only=True)
model.set_unet_particle_batch_size(NUM_PARTICLES)
positive, negative, real_tokens = model.encode_prompt(PROMPT, NEGATIVE_PROMPT)
initial_latents = model.make_initial_latents(
    NUM_PARTICLES, HEIGHT, WIDTH, INITIAL_SEED)
metrics = PromptMetrics()
print('initial latents:', tuple(initial_latents.shape))
print('real prompt tokens:', real_tokens)

## Pullback Basis And Particle-Specific Rho

This cell is needed only when `rho_star` is selected. It computes one initial basis, probes all rho candidates at one intermediate timestep, and keeps the CLIP-constrained assignment with the largest pairwise DINO distance.

In [ ]:
basis = None
selection = None
if 'rho_star' in SELECTED_METHODS:
    anchor, basis_t, _ = directions.compute_clean_anchor(
        initial_latents[:1], positive, negative, DDIM_STEPS,
        GUIDANCE_SCALE, ETA, ETA_SEED, BASIS_TIMESTEP,
    )
    basis, eigenvalues = pullback_basis.compute_pullback_basis(
        anchor, basis_t, positive, real_tokens,
        rank=PULLBACK_RANK,
        number_of_iterations=PULLBACK_ITERATIONS,
        seed=515,
        finite_difference_epsilon=0.5,
    )
    probe_latents, probe_t, _ = directions.compute_clean_anchor(
        initial_latents, positive, negative, DDIM_STEPS,
        GUIDANCE_SCALE, ETA, ETA_SEED, PROBE_TIMESTEP,
    )
    full_directions = rho_star.make_full_scale_directions(
        basis, positive, real_tokens, NUM_PARTICLES, 'disjoint', 777
    )
    probe = rho_star.probe_rho_candidates(
        probe_latents, probe_t, positive, negative, real_tokens,
        full_directions, [0.0] + CANDIDATE_RHOS, GUIDANCE_SCALE,
        schedule_start=999, schedule_end=500, schedule_power=2.0,
    )
    probe = rho_star.add_clip_dino_probe_features(
        probe, PROMPT, metrics, decode_batch_size=5
    )
    selection = rho_star.select_rho_combination_clip_dino(
        probe, max_clip_drop=MAX_CLIP_DROP,
        selectable_rhos=CANDIDATE_RHOS, search_strategy='beam',
        beam_width=4096, constraint_fallback='minimum_selectable',
    )
    print('eigenvalues:', [round(float(x), 4) for x in eigenvalues[:8]])
    print('selected rhos:', selection['selected_rhos'])

## Run Selected Methods

In [ ]:
images_by_method = {}

if 'clean_ddim' in SELECTED_METHODS:
    latents = ddim.sample_clean_ddim(
        initial_latents, positive, negative, DDIM_STEPS,
        GUIDANCE_SCALE, ETA, ETA_SEED,
    )
    images_by_method['clean_ddim'] = model.decode_latents(latents)

if 'cads' in SELECTED_METHODS:
    latents = cads.sample_cads(
        initial_latents, positive, negative, DDIM_STEPS,
        GUIDANCE_SCALE, ETA, ETA_SEED, start=900, end=600,
        noise_scale=0.15, psi=1.0, noise_seed=999,
        persistence='fresh', use_rescale=True,
    )
    images_by_method['cads'] = model.decode_latents(latents)

if 'tpso' in SELECTED_METHODS:
    optimized = tpso.optimize_token_offsets(
        PROMPT, NUM_PARTICLES, kappa=0.80, sigma=0.01,
        diversity_weight=1.0, learning_rate=1e-3,
        max_steps=200, min_steps=50, patience=15, seed=3407,
    )
    latents, _ = tpso.sample_tpso(
        initial_latents, positive, negative, optimized,
        DDIM_STEPS, GUIDANCE_SCALE, ETA, ETA_SEED, ratio=0.4,
    )
    images_by_method['tpso'] = model.decode_latents(latents)

if 'rho_star' in SELECTED_METHODS:
    latents, pullback_details = sample_adaptive_pullback(
        initial_latents, positive, negative, real_tokens, basis,
        DDIM_STEPS, GUIDANCE_SCALE, ETA, ETA_SEED,
        particle_rho=selection['selected_rhos'],
        start=999, end=500, schedule_power=2.0, mode='disjoint',
        direction_seed=777, number_of_refreshes=2,
        intermediate_rank=20, intermediate_iterations=1,
        intermediate_seed=1515, transition_steps=1,
        finite_difference_epsilon=0.5,
    )
    images_by_method['rho_star'] = model.decode_latents(latents)

In [ ]:
rows = []
for method_name, images in images_by_method.items():
    values, _ = metrics.compute(images, PROMPT)
    rows.append({'method': method_name, **values})
display(pd.DataFrame(rows)[
    ['method', 'clip', 'dino_sim_mean', 'dino_sim_max', 'mss', 'vendi']
])

fig, axes = plt.subplots(
    len(images_by_method), NUM_PARTICLES,
    figsize=(3.2 * NUM_PARTICLES, 3.3 * len(images_by_method)),
    squeeze=False,
)
for row, (method_name, images) in enumerate(images_by_method.items()):
    for particle, image in enumerate(images):
        axes[row, particle].imshow(image)
        axes[row, particle].set_title(f'{method_name} p{particle}')
        axes[row, particle].axis('off')
plt.tight_layout()
plt.show()